In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [2]:
df = yf.download("RELIANCE.NS", start="2011-01-01", end="2024-01-01",auto_adjust=True)
df.head()
reliance = yf.Ticker("RELIANCE.NS")
df = df.droplevel("Ticker", axis=1)
train = df[:"2020-12-31"]  # Jan 2011 – Dec 2020 (10 years)
test  = df["2021-01-01":]  # Jan 2021 – Jan 2024  (3 years)

[*********************100%***********************]  1 of 1 completed


In [3]:
def create_target(data, horizon=5):
    # Binary target: 1 if Close(T+horizon) > Close(T), else 0.
    if isinstance(data.columns, pd.MultiIndex):
        data = data.droplevel("Ticker", axis=1)
    close = data["Close"]
    future_close = close.shift(-horizon)
    target = (future_close > close).astype(int)
    target.name = "target"
    return target


In [4]:
def build_features(data):
    """
    Engineer features from OHLCV data for logistic regression.
    Returns a DataFrame of features aligned with the original index.
    """
    # Flatten MultiIndex columns if present (yfinance returns multi-level cols)
    if isinstance(data.columns, pd.MultiIndex):
        data = data.droplevel("Ticker", axis=1)

    close = data["Close"]
    volume = data["Volume"]

    feat = pd.DataFrame(index=data.index)

    # Price returns
    feat["ret_1"]  = close.pct_change(1)    # 1-day return
    feat["ret_5"]  = close.pct_change(5)    # 5-day return
    feat["ret_10"] = close.pct_change(10)   # 10-day return
    feat["ret_20"] = close.pct_change(20)   # 20-day return

    # Moving averages (ratio to current price)
    feat["ma5_ratio"]  = close / close.rolling(5).mean()
    feat["ma10_ratio"] = close / close.rolling(10).mean()
    feat["ma20_ratio"] = close / close.rolling(20).mean()
    feat["ma50_ratio"] = close / close.rolling(50).mean()

    # Volatility (rolling std of returns)
    feat["vol_5"]  = close.pct_change().rolling(5).std()
    feat["vol_20"] = close.pct_change().rolling(20).std()

    # RSI (14-day)
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss
    feat["rsi_14"] = 100 - (100 / (1 + rs))

    # Volume change
    feat["vol_change_1"] = volume.pct_change(1)
    feat["vol_change_5"] = volume.pct_change(5)

    # Rolling Z-scores (how far price is from recent average, in std devs)
    feat["zscore_20"] = (close - close.rolling(20).mean()) / close.rolling(20).std()
    feat["zscore_50"] = (close - close.rolling(50).mean()) / close.rolling(50).std()

    # Replace any inf values with NaN (e.g. from zero-volume days)
    feat.replace([np.inf, -np.inf], np.nan, inplace=True)

    return feat

In [5]:
def run_logistic_regression(train_df, test_df, horizon=5):
    """
    Train a Logistic Regression model on train_df and evaluate on test_df.
    Predicts whether Close(T+5) > Close(T).
    """
    # Build features & target for both sets
    X_train = build_features(train_df)
    y_train = create_target(train_df, horizon)

    X_test = build_features(test_df)
    y_test = create_target(test_df, horizon)

    # Align and drop rows with NaN (from rolling windows / shift)
    train_combined = pd.concat([X_train, y_train], axis=1).dropna()
    test_combined  = pd.concat([X_test, y_test], axis=1).dropna()

    X_train_clean = train_combined.drop("target", axis=1)
    y_train_clean = train_combined["target"]
    X_test_clean  = test_combined.drop("target", axis=1)
    y_test_clean  = test_combined["target"]

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_clean)
    X_test_scaled  = scaler.transform(X_test_clean)

    # Train Logistic Regression
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train_clean)

    # Predict
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred  = model.predict(X_test_scaled)

    # ---- Results ----
    print("\n" + "=" * 60)
    print("  LOGISTIC REGRESSION – Predict Close(T+5) > Close(T)")
    print("=" * 60)

    print(f"\n--- Train Set Results ---")
    print(f"Accuracy: {accuracy_score(y_train_clean, y_train_pred):.4f}")
    print(classification_report(y_train_clean, y_train_pred, target_names=["Down/Flat", "Up"]))

    print(f"--- Test Set Results ---")
    print(f"Accuracy: {accuracy_score(y_test_clean, y_test_pred):.4f}")
    print(classification_report(y_test_clean, y_test_pred, target_names=["Down/Flat", "Up"]))

    print("Confusion Matrix (Test):")
    print(confusion_matrix(y_test_clean, y_test_pred))

    # Feature importance (coefficient magnitudes)
    feature_names = X_train_clean.columns
    coefs = pd.Series(model.coef_[0], index=feature_names).sort_values(key=abs, ascending=False)
    print("\nTop Feature Coefficients:")
    print(coefs.to_string())

    return model, scaler

In [6]:
model, scaler = run_logistic_regression(train, test, horizon=5)


  LOGISTIC REGRESSION – Predict Close(T+5) > Close(T)

--- Train Set Results ---
Accuracy: 0.5577
              precision    recall  f1-score   support

   Down/Flat       0.54      0.31      0.39      1120
          Up       0.56      0.77      0.65      1290

    accuracy                           0.56      2410
   macro avg       0.55      0.54      0.52      2410
weighted avg       0.55      0.56      0.53      2410

--- Test Set Results ---
Accuracy: 0.5202
              precision    recall  f1-score   support

   Down/Flat       0.48      0.31      0.37       324
          Up       0.54      0.71      0.61       368

    accuracy                           0.52       692
   macro avg       0.51      0.51      0.49       692
weighted avg       0.51      0.52      0.50       692

Confusion Matrix (Test):
[[ 99 225]
 [107 261]]

Top Feature Coefficients:
ma10_ratio     -0.306073
ma20_ratio      0.223590
vol_5           0.132053
ret_10          0.130863
zscore_50      -0.119251
ma5_r

In [7]:
# ---- Trading Simulation (Long & Short) ----
def run_trading_simulation(test_df, model, scaler, initial_capital=1_000_000):
    """Simulate trading: Long on signal=1, Short on signal=0."""
    X_test = build_features(test_df)
    X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
    valid_idx = X_test.dropna().index
    X_test_clean = X_test.loc[valid_idx]
    X_test_scaled = scaler.transform(X_test_clean)
    predictions = pd.Series(model.predict(X_test_scaled), index=valid_idx)

    close = test_df["Close"]
    open_price = test_df["Open"]
    capital = initial_capital
    trades = []
    i = 0
    valid_dates = valid_idx.tolist()

    while i < len(valid_dates):
        signal_date = valid_dates[i]
        signal = predictions[signal_date]
        signal_pos = test_df.index.get_loc(signal_date)
        entry_pos = signal_pos + 1
        exit_pos  = signal_pos + 5
        if exit_pos >= len(test_df):
            break
        entry_date = test_df.index[entry_pos]
        exit_date  = test_df.index[exit_pos]
        entry_price = open_price.iloc[entry_pos]
        exit_price  = close.iloc[exit_pos]

        if signal == 1:
            trade_return = (exit_price / entry_price) - 1
            direction = "LONG"
        else:
            trade_return = (entry_price / exit_price) - 1
            direction = "SHORT"

        new_capital = capital * (1 + trade_return)
        trades.append({
            "signal_date": signal_date, "direction": direction,
            "entry_date": entry_date, "exit_date": exit_date,
            "entry_price": round(entry_price, 2),
            "exit_price": round(exit_price, 2),
            "trade_return": round(trade_return, 4),
            "capital_before": round(capital, 2),
            "capital_after": round(new_capital, 2),
        })
        capital = new_capital
        i += 1
        while i < len(valid_dates) and valid_dates[i] <= exit_date:
            i += 1

    return pd.DataFrame(trades), capital


def compute_financial_metrics(trades_df, initial_capital, risk_free_rate=0.05):
    """Compute Total Return and annualized Sharpe Ratio."""
    final_capital = trades_df["capital_after"].iloc[-1]
    total_return = (final_capital / initial_capital) - 1
    trade_returns = trades_df["trade_return"]
    trades_per_year = 250 / 5
    rf_per_trade = risk_free_rate / trades_per_year
    sharpe = (trade_returns.mean() - rf_per_trade) / trade_returns.std() * np.sqrt(trades_per_year)
    return total_return, sharpe, final_capital


# ---- Run Simulation ----
INITIAL_CAPITAL = 1_000_000
trades_df, final_capital = run_trading_simulation(test, model, scaler, INITIAL_CAPITAL)
total_return, sharpe, _ = compute_financial_metrics(trades_df, INITIAL_CAPITAL)

print("Initial Capital:", f"Rs.{INITIAL_CAPITAL:,.2f}")
print("Final Capital:  ", f"Rs.{final_capital:,.2f}")
print("Total Return:   ", f"{total_return * 100:.2f}%")
print("Sharpe Ratio:   ", round(sharpe, 4))
print("Total Trades:   ", len(trades_df))
long_count = (trades_df["direction"]=="LONG").sum()
short_count = (trades_df["direction"]=="SHORT").sum()
print("Long / Short:   ", long_count, "/", short_count)
win_rate = (trades_df["trade_return"] > 0).mean() * 100
print("Win Rate:       ", f"{win_rate:.1f}%")
print("\nFirst 10 trades:")
trades_df.head(10)

Initial Capital: Rs.1,000,000.00
Final Capital:   Rs.877,891.26
Total Return:    -12.21%
Sharpe Ratio:    -0.344
Total Trades:    115
Long / Short:    83 / 32
Win Rate:        46.1%

First 10 trades:


,signal_date,direction,entry_date,exit_date,entry_price,exit_price,trade_return,capital_before,capital_after
0,2021-03-15,LONG,2021-03-16,2021-03-22,958.55,935.16,-0.0244,1000000.00,975591.28
1,2021-03-23,LONG,2021-03-24,2021-03-31,943.02,908.27,-0.0369,975591.28,939635.54
2,2021-04-01,LONG,2021-04-05,2021-04-09,918.18,898.72,-0.0212,939635.54,919728.78
3,2021-04-12,LONG,2021-04-13,2021-04-20,872.40,862.04,-0.0119,919728.78,908805.80
4,2021-04-22,LONG,2021-04-23,2021-04-29,864.24,917.77,0.0619,908805.80,965093.69
5,2021-04-30,LONG,2021-05-03,2021-05-07,891.45,875.92,-0.0174,965093.69,948280.61
6,2021-05-10,LONG,2021-05-11,2021-05-18,868.32,901.40,0.0381,948280.61,984404.46
7,2021-05-19,LONG,2021-05-20,2021-05-26,905.91,893.28,-0.0139,984404.46,970682.19
8,2021-05-27,LONG,2021-05-28,2021-06-03,902.33,1001.92,0.1104,970682.19,1077823.08
9,2021-06-04,LONG,2021-06-07,2021-06-11,998.46,1007.01,0.0086,1077823.08,1087057.27
